# Fine-tune VideoMAE nhận diện hành vi nghi trộm cắp (demo đồ án)

Notebook này fine-tune `MCG-NJU/videomae-base-finetuned-kinetics` (VideoMAE đã học nhận diện hành động trên Kinetics-400) thành bộ phân loại hai lớp **normal** và **shoplifting** trên clip bạn tự quay, rồi lưu một "bundle" để chạy thời gian thực bằng `demo_videomae.py`.

**Giấy phép:** model gốc theo CC BY-NC 4.0, chỉ dùng phi thương mại. Dùng cho đồ án trường thì ổn, nhưng không đưa vào sản phẩm bán ra.

## Cách quay dữ liệu (quyết định chất lượng demo)

Cấu trúc thư mục:

```
clips/
  normal/        an_001.mp4  an_002.mp4  binh_001.mp4 ...
  shoplifting/   an_101.mp4  binh_101.mp4 ...
```

- **Tên file bắt đầu bằng mã người quay** (`an_`, `binh_`). Notebook chia train/test **theo người**, để model không "nhớ mặt" người đã thấy khi huấn luyện. Cần ít nhất 4 người.
- **Mỗi lớp 40–60 clip trở lên**, mỗi clip 3–5 giây, cùng một góc camera cố định như lúc demo, quay nhiều ánh sáng và nhiều vị trí kệ.
- **shoplifting** là cảnh **diễn lại**: giấu món hàng (hoặc vật giả) vào túi áo, túi quần, túi xách.
- **normal** phải có cả "âm tính khó": bỏ hàng vào giỏ, cúi nhặt đồ, lấy hàng trên kệ, xem điện thoại, đút tay vào túi không lấy gì, hai người đứng gần nhau. Thiếu phần này model sẽ báo nhầm ở mọi động tác đưa tay.
- Chỉ quay người đã đồng ý tham gia.

In [ ]:
%pip install -q transformers opencv-python-headless scikit-learn

import os, sys, json, random, shutil, time, importlib
from pathlib import Path
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, precision_recall_fscore_support, confusion_matrix

CLASSES = ["normal", "shoplifting"]

def _find_root():
    """Tìm thư mục chứa cả normal/ và shoplifting/."""
    for base in (os.environ.get("DATA_ROOT"), "/kaggle/input", "/content/drive/MyDrive", "clips", "."):
        if base and Path(base).exists():
            p = Path(base)
            if all((p / c).is_dir() for c in CLASSES):
                return str(p)
            for sub in p.rglob("*"):
                if sub.is_dir() and all((sub / c).is_dir() for c in CLASSES):
                    return str(sub)
    return "clips"

CFG = dict(
    base=os.environ.get("BASE", "MCG-NJU/videomae-base-finetuned-kinetics"),
    data_root=_find_root(),
    out_dir=os.environ.get("OUT", "/kaggle/working/checkpoints/videomae_v1" if Path("/kaggle/working").exists()
                           else "checkpoints/videomae_v1"),
    epochs=int(os.environ.get("EPOCHS", 8)), batch=int(os.environ.get("BATCH", 4)), accum=2,
    lr=float(os.environ.get("LR", 3e-5)), weight_decay=0.05, warmup_frac=0.1,
    test_group_frac=0.25, val_group_frac=0.2,
    grad_ckpt=os.environ.get("GRAD_CKPT", "1") == "1",
    workers=int(os.environ.get("WORKERS", 2)), seed=0, threshold_mode="f1",
)
random.seed(CFG["seed"]); np.random.seed(CFG["seed"]); torch.manual_seed(CFG["seed"])
device = "cuda" if torch.cuda.is_available() else "cpu"
print("thiết bị:", device, "| dữ liệu:", CFG["data_root"], "| base:", CFG["base"])

In [ ]:
%%writefile videomae_lite.py
"""Tiện ích VideoMAE cho demo phát hiện hành vi nghi trộm cắp.

Dùng chung cho notebook huấn luyện và script demo thời gian thực:
  - đọc clip từ file video (chỉ giữ các khung cần thiết, tiết kiệm RAM)
  - tiền xử lý (resize, chuẩn hóa) và tăng cường dữ liệu cho huấn luyện
  - Predictor: nạp model đã lưu và trả xác suất "shoplifting" cho một cửa sổ khung hình
"""
from __future__ import annotations

import json
from pathlib import Path

import cv2
import numpy as np
import torch

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm", ".m4v"}


def uniform_indices(n_total: int, n_frames: int = 16, offset: float = 0.0,
                    span: float = 1.0) -> np.ndarray:
    """n_frames chỉ số đều nhau trong một đoạn con của [0, n_total).

    span   : tỉ lệ độ dài đoạn con so với cả clip (1.0 = cả clip).
    offset : vị trí bắt đầu của đoạn con (0 = đầu, 1 = sát cuối).
    """
    length = max(1, int(round(n_total * span)))
    start = int(round((n_total - length) * offset))
    return np.linspace(start, start + length - 1, n_frames).round().astype(int)


def clip_info(path) -> tuple[int, float]:
    """(số khung hình, fps) của một file video."""
    cap = cv2.VideoCapture(str(path))
    n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps = float(cap.get(cv2.CAP_PROP_FPS)) or 30.0
    cap.release()
    return n, fps


def read_video_frames(path, n_frames: int = 16, size: int = 224, train: bool = False,
                      rng: np.random.Generator | None = None) -> np.ndarray:
    """Đọc n_frames khung hình đều nhau -> uint8 [T, size, size, 3] (RGB).

    Khi train=True lấy ngẫu nhiên một đoạn con 70-100% của clip để đa dạng hóa nhịp độ.
    """
    rng = rng or np.random.default_rng()
    n_total, _ = clip_info(path)
    if n_total <= 0:
        raise ValueError(f"Không đọc được số khung hình: {path}")
    span, offset = (float(rng.uniform(0.7, 1.0)), float(rng.uniform(0, 1))) if train else (1.0, 0.0)
    idx = uniform_indices(n_total, n_frames, offset, span)
    need = set(idx.tolist())
    cap = cv2.VideoCapture(str(path))
    got, last = {}, None
    for i in range(int(idx.max()) + 1):
        if not cap.grab():
            break
        if i in need:
            ok, f = cap.retrieve()
            if ok:
                last = cv2.cvtColor(cv2.resize(f, (size, size), interpolation=cv2.INTER_AREA),
                                    cv2.COLOR_BGR2RGB)
                got[i] = last
    cap.release()
    if last is None:
        raise ValueError(f"Không giải mã được khung nào: {path}")
    return np.stack([got.get(int(i), last) for i in idx])   # thiếu khung -> lặp khung cuối


def augment(frames: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    """Tăng cường nhẹ, áp dụng giống nhau cho mọi khung của clip."""
    t, h, w, _ = frames.shape
    if rng.random() < 0.5:                                   # lật gương
        frames = frames[:, :, ::-1]
    s = float(rng.uniform(0.8, 1.0))                         # cắt ngẫu nhiên rồi phóng lại
    ch, cw = int(h * s), int(w * s)
    y0, x0 = int(rng.integers(0, h - ch + 1)), int(rng.integers(0, w - cw + 1))
    crop = frames[:, y0:y0 + ch, x0:x0 + cw]
    frames = np.stack([cv2.resize(np.ascontiguousarray(f), (w, h), interpolation=cv2.INTER_LINEAR)
                       for f in crop])
    alpha, beta = float(rng.uniform(0.8, 1.2)), float(rng.uniform(-20, 20))   # tương phản, sáng
    return np.clip(frames.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)


def to_tensor(frames: np.ndarray, mean=IMAGENET_MEAN, std=IMAGENET_STD) -> torch.Tensor:
    """uint8 [T,H,W,3] -> float [T,3,H,W] đã chuẩn hóa (đúng dạng pixel_values của VideoMAE)."""
    x = torch.from_numpy(np.ascontiguousarray(frames)).float() / 255.0
    x = (x - torch.tensor(mean)) / torch.tensor(std)
    return x.permute(0, 3, 1, 2).contiguous()


def resize_rgb(frame_bgr: np.ndarray, size: int) -> np.ndarray:
    return cv2.cvtColor(cv2.resize(frame_bgr, (size, size), interpolation=cv2.INTER_AREA),
                        cv2.COLOR_BGR2RGB)


class Predictor:
    """Nạp bundle (model/ + preprocess.json + calibration.json) và chấm điểm một cửa sổ."""

    def __init__(self, bundle_dir, device: str | None = None):
        from transformers import VideoMAEForVideoClassification
        d = Path(bundle_dir)
        self.meta = json.loads((d / "preprocess.json").read_text(encoding="utf-8"))
        self.cal = json.loads((d / "calibration.json").read_text(encoding="utf-8"))
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = VideoMAEForVideoClassification.from_pretrained(d / "model").to(self.device).eval()
        self.pos = int(self.model.config.label2id["shoplifting"])
        self.n_frames = int(self.meta["num_frames"])
        self.size = int(self.meta["size"])
        self.mean, self.std = self.meta["mean"], self.meta["std"]
        self.threshold = float(self.cal["threshold"])
        self.clip_sec = float(self.meta["clip_sec"])

    @torch.no_grad()
    def predict(self, frames_rgb: np.ndarray) -> float:
        """frames_rgb: uint8 [n_frames, size, size, 3]. Trả xác suất 'shoplifting'."""
        x = to_tensor(frames_rgb, self.mean, self.std).unsqueeze(0).to(self.device)
        return float(torch.softmax(self.model(pixel_values=x).logits, -1)[0, self.pos])


In [ ]:
%%writefile demo_videomae.py
#!/usr/bin/env python3
"""Demo thời gian thực: VideoMAE phát hiện hành vi nghi trộm cắp, gửi cảnh báo qua MQTT.

Ví dụ:
    python demo_videomae.py --bundle checkpoints/videomae_v1 --source 0 --show
    python demo_videomae.py --bundle checkpoints/videomae_v1 --source clip.mp4 --no-mqtt
    python demo_videomae.py --bundle checkpoints/videomae_v1 --source rtsp://... --mqtt-host localhost

Chỉ CẢNH BÁO để người xem xét, không kết luận "ăn trộm".
"""
from __future__ import annotations

import argparse
import json
import time
import uuid
from collections import deque
from pathlib import Path

import cv2
import numpy as np

from videomae_lite import Predictor, resize_rgb, uniform_indices


def make_mqtt(args):
    """Kết nối MQTT. Trả None nếu tắt hoặc không kết nối được (demo vẫn chạy)."""
    if args.no_mqtt:
        return None
    try:
        import paho.mqtt.client as mqtt
        try:
            client = mqtt.Client(mqtt.CallbackAPIVersion.VERSION2, client_id=f"cam-{args.cam_id}")
        except AttributeError:                      # paho-mqtt 1.x
            client = mqtt.Client(client_id=f"cam-{args.cam_id}")
        status = f"store/{args.store_id}/cam/{args.cam_id}/status"
        client.will_set(status, "offline", qos=1, retain=True)   # báo offline nếu mất kết nối
        client.connect(args.mqtt_host, args.mqtt_port, keepalive=30)
        client.loop_start()
        client.publish(status, "online", qos=1, retain=True)
        print(f"[mqtt] đã kết nối {args.mqtt_host}:{args.mqtt_port}")
        return client
    except Exception as e:                          # noqa: BLE001
        print(f"[mqtt] KHÔNG kết nối được ({e}). Tiếp tục không gửi MQTT.")
        return None


def save_clip(frames_bgr, fps, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    h, w = frames_bgr[0].shape[:2]
    out = cv2.VideoWriter(str(path), cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for f in frames_bgr:
        out.write(f)
    out.release()


def main() -> None:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--bundle", required=True, help="thư mục bundle (có model/, preprocess.json, calibration.json)")
    ap.add_argument("--source", default="0", help="số camera (0, 1...), đường dẫn video hoặc URL RTSP")
    ap.add_argument("--hop-sec", type=float, default=1.0, help="bước trượt giữa hai lần chấm điểm")
    ap.add_argument("--threshold", type=float, default=None, help="ghi đè ngưỡng trong bundle")
    ap.add_argument("--k", type=int, default=2, help="số cửa sổ liên tiếp vượt ngưỡng mới báo")
    ap.add_argument("--cooldown", type=float, default=10.0, help="giây chờ giữa hai cảnh báo")
    ap.add_argument("--store-id", default="demo")
    ap.add_argument("--cam-id", default="1")
    ap.add_argument("--mqtt-host", default="localhost")
    ap.add_argument("--mqtt-port", type=int, default=1883)
    ap.add_argument("--no-mqtt", action="store_true")
    ap.add_argument("--clips-dir", default="data/clips_out")
    ap.add_argument("--show", action="store_true", help="hiện cửa sổ video (cần màn hình)")
    args = ap.parse_args()

    pred = Predictor(args.bundle)
    thr = pred.threshold if args.threshold is None else args.threshold
    print(f"[model] thiết bị {pred.device} | cửa sổ {pred.clip_sec:.1f}s, {pred.n_frames} khung | ngưỡng {thr:.3f} | K={args.k}")

    src = int(args.source) if args.source.isdigit() else args.source
    live = isinstance(src, int) or str(src).startswith(("rtsp://", "http://", "https://"))
    cap = cv2.VideoCapture(src)
    if not cap.isOpened():
        raise SystemExit(f"Không mở được nguồn video: {args.source}")
    fps = cap.get(cv2.CAP_PROP_FPS) or 15.0
    client = make_mqtt(args)
    topic = f"store/{args.store_id}/cam/{args.cam_id}/alert"

    buf: deque = deque()                 # (thời điểm, khung BGR đã thu nhỏ)
    next_eval, streak, last_alert = None, 0, -1e9
    idx, t_start, n_alerts = 0, time.monotonic(), 0
    p_last = 0.0

    while True:
        ok, frame = cap.read()
        if not ok:
            break
        t = (time.monotonic() - t_start) if live else idx / fps
        idx += 1
        h, w = frame.shape[:2]
        scale = 640.0 / max(h, w)
        small = cv2.resize(frame, (int(w * scale), int(h * scale))) if scale < 1 else frame
        buf.append((t, small))
        while buf and t - buf[0][0] > pred.clip_sec:
            buf.popleft()
        if next_eval is None:
            next_eval = buf[0][0] + pred.clip_sec
        if t >= next_eval and (buf[-1][0] - buf[0][0]) >= 0.9 * pred.clip_sec:
            next_eval = t + args.hop_sec
            sel = uniform_indices(len(buf), pred.n_frames)
            window = [buf[i][1] for i in sel]
            p_last = pred.predict(np.stack([resize_rgb(f, pred.size) for f in window]))
            streak = streak + 1 if p_last >= thr else 0
            print(f"[{t:7.1f}s] p(nghi trộm)={p_last:.3f} | liên tiếp {streak}/{args.k}")
            if streak >= args.k and (t - last_alert) >= args.cooldown:
                last_alert, n_alerts = t, n_alerts + 1
                eid = uuid.uuid4().hex[:12]
                clip_path = Path(args.clips_dir) / f"{args.store_id}_{args.cam_id}_{eid}.mp4"
                save_clip([f for _, f in buf], max(1.0, len(buf) / max(1e-6, buf[-1][0] - buf[0][0])), clip_path)
                msg = dict(event_id=eid, ts=time.strftime("%Y-%m-%dT%H:%M:%S%z"), cam_id=args.cam_id,
                           score=round(p_last, 4), threshold=round(thr, 4), clip_path=str(clip_path))
                print(f"  >>> CẢNH BÁO nghi vấn: {msg}")
                if client is not None:
                    client.publish(topic, json.dumps(msg), qos=1)
        if args.show:
            vis = frame.copy()
            cv2.putText(vis, f"p={p_last:.2f} thr={thr:.2f} alerts={n_alerts}", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0, 0, 255) if streak else (0, 200, 0), 2)
            cv2.imshow("demo", vis)
            if cv2.waitKey(1) & 0xFF == ord("q"):
                break

    cap.release()
    if args.show:
        cv2.destroyAllWindows()
    if client is not None:
        client.publish(f"store/{args.store_id}/cam/{args.cam_id}/status", "offline", qos=1, retain=True)
        client.loop_stop(); client.disconnect()
    print(f"Xong. Số cảnh báo: {n_alerts}")


if __name__ == "__main__":
    main()


In [ ]:
importlib.invalidate_caches(); sys.modules.pop("videomae_lite", None)   # luôn nạp bản mới ghi
import videomae_lite as vl
from videomae_lite import VIDEO_EXTS, read_video_frames, augment, to_tensor, clip_info

root = Path(CFG["data_root"])
items = []                                   # (đường dẫn, nhãn, nhóm = mã người)
for label, cname in enumerate(CLASSES):
    for p in sorted((root / cname).rglob("*")):
        if p.suffix.lower() in VIDEO_EXTS:
            items.append((p, label, p.stem.split("_")[0]))
assert items, f"Không thấy video trong {root}/{CLASSES}. Kiểm tra cấu trúc thư mục."

n_cls = [sum(1 for _, y, _ in items if y == k) for k in (0, 1)]
groups = sorted({g for _, _, g in items})
print(f"{len(items)} clip | normal {n_cls[0]} | shoplifting {n_cls[1]} | {len(groups)} nhóm (người): {groups}")
if min(n_cls) < 30:
    print("CẢNH BÁO: mỗi lớp nên có từ 40 clip trở lên, kết quả sẽ rất nhiễu nếu ít hơn.")

# Chia theo nhóm (người) để tránh rò rỉ; nếu quá ít nhóm thì chia theo clip và cảnh báo.
rng = random.Random(CFG["seed"])
if len(groups) >= 4:
    gs = groups[:]; rng.shuffle(gs)
    n_te = max(1, round(len(gs) * CFG["test_group_frac"]))
    test_g, rest = set(gs[:n_te]), gs[n_te:]
    n_va = max(1, round(len(rest) * CFG["val_group_frac"]))
    val_g, train_g = set(rest[:n_va]), set(rest[n_va:])
    pick = lambda S: [it for it in items if it[2] in S]
    train_items, val_items, test_items = pick(train_g), pick(val_g), pick(test_g)
else:
    print("CẢNH BÁO: dưới 4 người nên phải chia ngẫu nhiên theo clip. Model có thể 'nhớ' người/cảnh, điểm sẽ cao ảo.")
    idx = list(range(len(items))); rng.shuffle(idx)
    a, b = int(len(idx) * 0.6), int(len(idx) * 0.8)
    train_items = [items[i] for i in idx[:a]]; val_items = [items[i] for i in idx[a:b]]; test_items = [items[i] for i in idx[b:]]

for name, S in (("train", train_items), ("val", val_items), ("test", test_items)):
    print(f"{name:5s}: {len(S):3d} clip | normal {sum(1 for _,y,_ in S if y==0)} | shoplifting {sum(1 for _,y,_ in S if y==1)} | người: {sorted({g for _,_,g in S})}")
for name, S in (("val", val_items), ("test", test_items)):
    if len({y for _, y, _ in S}) < 2:
        print(f"CẢNH BÁO: tập {name} thiếu một lớp, các chỉ số như AUC sẽ không tính được. Quay thêm dữ liệu hoặc đổi seed.")

In [ ]:
from transformers import VideoMAEForVideoClassification

model = VideoMAEForVideoClassification.from_pretrained(
    CFG["base"], num_labels=2, id2label={0: "normal", 1: "shoplifting"},
    label2id={"normal": 0, "shoplifting": 1}, ignore_mismatched_sizes=True).to(device)
if CFG["grad_ckpt"] and device == "cuda":
    model.gradient_checkpointing_enable()
N_FR, SIZE = int(model.config.num_frames), int(model.config.image_size)
try:
    from transformers import VideoMAEImageProcessor
    _p = VideoMAEImageProcessor.from_pretrained(CFG["base"])
    MEAN, STD = list(_p.image_mean), list(_p.image_std)
except Exception:
    MEAN, STD = list(vl.IMAGENET_MEAN), list(vl.IMAGENET_STD)
print(f"model: {N_FR} khung x {SIZE}px | mean {MEAN} | std {STD}")

class ClipDS(Dataset):
    def __init__(self, items, train):
        self.items, self.train = items, train
    def __len__(self):
        return len(self.items)
    def __getitem__(self, i):
        path, y, _ = self.items[i]
        rng = np.random.default_rng((CFG["seed"] * 1_000_003 + i + (int(time.time() * 1000) if self.train else 0)) % 2**32)
        fr = read_video_frames(path, N_FR, SIZE, train=self.train, rng=rng)
        if self.train:
            fr = augment(fr, rng)
        return to_tensor(fr, MEAN, STD), y

def loader(items, train):
    return DataLoader(ClipDS(items, train), batch_size=CFG["batch"], shuffle=train, drop_last=False,
                      num_workers=CFG["workers"], pin_memory=(device == "cuda"))

@torch.no_grad()
def predict_probs(items):
    model.eval(); P, Y = [], []
    for x, y in loader(items, False):
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            lg = model(pixel_values=x.to(device)).logits
        P.append(torch.softmax(lg.float(), -1)[:, 1].cpu().numpy()); Y.append(y.numpy())
    return (np.concatenate(P), np.concatenate(Y)) if P else (np.zeros(0), np.zeros(0, int))

def safe_auc(y, p):
    return float(roc_auc_score(y, p)) if len(set(y.tolist())) == 2 else float("nan")

In [ ]:
# ---- Huấn luyện
cnt = np.bincount([y for _, y, _ in train_items], minlength=2).astype(float)
class_w = torch.tensor(cnt.sum() / (2 * np.maximum(cnt, 1)), dtype=torch.float32, device=device)  # cân bằng lớp
print("trọng số lớp:", class_w.tolist())

opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
steps_per_epoch = max(1, len(loader(train_items, True)) // CFG["accum"])
total, warm = CFG["epochs"] * steps_per_epoch, max(1, int(CFG["warmup_frac"] * CFG["epochs"] * steps_per_epoch))
sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: (s + 1) / warm if s < warm else 0.5 * (1 + np.cos(np.pi * (s - warm) / max(1, total - warm))))
scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
crit = nn.CrossEntropyLoss(weight=class_w)

best_auc, best_state, hist = -1.0, None, []
t0 = time.time()
for ep in range(1, CFG["epochs"] + 1):
    model.train(); run = n = 0; opt.zero_grad()
    for step, (x, y) in enumerate(loader(train_items, True), 1):
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
            loss = crit(model(pixel_values=x).logits.float(), y) / CFG["accum"]
        scaler.scale(loss).backward()
        if step % CFG["accum"] == 0:
            scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt); scaler.update(); opt.zero_grad(); sched.step()
        run += loss.item() * CFG["accum"] * len(y); n += len(y)
    pv, yv = predict_probs(val_items)
    auc_v = safe_auc(yv, pv); acc_v = float(((pv >= 0.5) == yv).mean()) if len(yv) else float("nan")
    hist.append(dict(epoch=ep, train_loss=run / n, val_auc=auc_v, val_acc=acc_v))
    print(f"epoch {ep:2d} | train loss {run/n:.4f} | val AUC {auc_v:.3f} | val acc {acc_v:.3f} | {time.time()-t0:.0f}s")
    score = auc_v if not np.isnan(auc_v) else -0.5 + ep * 1e-3         # val thiếu lớp: lấy epoch cuối
    if score > best_auc:
        best_auc, best_state = score, {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
model.load_state_dict(best_state); model.eval()
print("đã nạp checkpoint tốt nhất theo AUC val")

In [ ]:
# ---- Đánh giá trên người chưa từng thấy (tập test) và chọn ngưỡng từ val
pv, yv = predict_probs(val_items)
pt, yt = predict_probs(test_items)

def pick_threshold(p, y, mode):
    if len(set(y.tolist())) < 2:
        return 0.5
    cands = np.unique(np.concatenate([[0.5], (np.sort(p)[1:] + np.sort(p)[:-1]) / 2]))
    best_t, best = 0.5, -1
    for t in cands:
        pr, rc, f1, _ = precision_recall_fscore_support(y, (p >= t).astype(int), average="binary", zero_division=0)
        s = f1 if mode == "f1" else (rc >= 0.9) * (1 - (np.mean(p[y == 0] >= t)))   # recall >= 0.9, ít báo nhầm nhất
        if s > best:
            best, best_t = s, float(t)
    return best_t

thr = pick_threshold(pv, yv, CFG["threshold_mode"])
pred = (pt >= thr).astype(int)
pr, rc, f1, _ = precision_recall_fscore_support(yt, pred, average="binary", zero_division=0)
metrics = dict(test_clips=int(len(yt)), auc=safe_auc(yt, pt), accuracy=float((pred == yt).mean()),
               precision=float(pr), recall=float(rc), f1=float(f1), threshold=thr)
print(json.dumps(metrics, indent=2))
print("ma trận nhầm lẫn (hàng: thật, cột: dự đoán; normal, shoplifting):")
print(confusion_matrix(yt, pred, labels=[0, 1]))

print(f"\n{'ngưỡng':>8}{'bắt được trộm (TPR)':>22}{'báo nhầm (FPR)':>18}")
ops = {}
for t in (0.3, 0.5, 0.7, 0.9, thr):
    tp = float((pt[yt == 1] >= t).mean()) if (yt == 1).any() else float("nan")
    fp = float((pt[yt == 0] >= t).mean()) if (yt == 0).any() else float("nan")
    ops[f"{t:.3f}"] = dict(tpr=tp, fpr=fp); print(f"{t:8.3f}{tp:22.3f}{fp:18.3f}")
if len(yt) < 20:
    print("\nLưu ý: tập test dưới 20 clip nên mọi con số ở trên dao động rất mạnh. Hãy quay thêm dữ liệu trước khi tin vào chúng.")

In [ ]:
# ---- Lưu bundle: model + tiền xử lý + ngưỡng + hai file mã
out = Path(CFG["out_dir"]); out.mkdir(parents=True, exist_ok=True)
model.save_pretrained(out / "model")
clip_secs = []
for p, _, _ in train_items:
    n, fps = clip_info(p)
    if n > 0:
        clip_secs.append(n / fps)
clip_sec = float(np.median(clip_secs)) if clip_secs else 3.0
json.dump(dict(num_frames=N_FR, size=SIZE, mean=MEAN, std=STD, clip_sec=clip_sec, base=CFG["base"],
               classes=CLASSES, note="Cửa sổ suy luận phải dài xấp xỉ clip_sec và lấy num_frames khung đều nhau."),
          open(out / "preprocess.json", "w"), indent=2, ensure_ascii=False)
json.dump(dict(threshold=thr, mode=CFG["threshold_mode"], operating_points_test=ops),
          open(out / "calibration.json", "w"), indent=2)
json.dump(dict(model="VideoMAE fine-tune (2 lớp)", base=CFG["base"], license_note="Model gốc CC BY-NC 4.0: chỉ dùng phi thương mại",
               metrics_test=metrics, history=hist,
               split=dict(train=[g for g in sorted({g for _,_,g in train_items})], val=sorted({g for _,_,g in val_items}), test=sorted({g for _,_,g in test_items})),
               clips=dict(train=len(train_items), val=len(val_items), test=len(test_items)),
               trained_at=time.strftime("%Y-%m-%d %H:%M")),
          open(out / "meta.json", "w"), indent=2, ensure_ascii=False)
for f in ("videomae_lite.py", "demo_videomae.py"):
    shutil.copy(f, out / f)
print("Đã lưu bundle vào", out.resolve(), f"| cửa sổ suy luận {clip_sec:.1f}s")

## Chạy demo thời gian thực

Tải thư mục bundle về máy, cài `pip install torch transformers opencv-python paho-mqtt`, rồi:

```bash
# bật broker MQTT (Docker)
docker run -d -p 1883:1883 eclipse-mosquitto

# webcam
python demo_videomae.py --bundle videomae_v1 --source 0 --show
# hoặc một file video, không cần MQTT
python demo_videomae.py --bundle videomae_v1 --source clip.mp4 --no-mqtt
```

Cảnh báo gửi tới topic `store/<store>/cam/<cam>/alert` dưới dạng JSON `{event_id, ts, cam_id, score, threshold, clip_path}`.
Cảnh báo chỉ có nghĩa là "nghi vấn, cần người xem lại".